In [1]:
import numpy as np
import pandas as pd
import sys

from pathlib import Path
from typing import Tuple

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.vis as vis

from config_handlers.eval_handler import EvalHandler
from config_handlers.rk_handler import RKHandler

In [2]:
rk_hdl = RKHandler()
eval_hdl = EvalHandler()

model_list = rk_hdl.get_model_list()
rk_hdl.set_curr_model(model_list[0])
ds_list = rk_hdl.get_ds_list()

all_ds = ds_list * 4
all_models = model_list * 4
all_models.sort()

MC_DATASETS = ["seed", "vqav2"]
OPEN_DATASETS = ["llava-bench", "mmbench"]

CHARTS_DIR = Path("behaviour_charts")
data_io.make_dir(CHARTS_DIR)

[Errno 17] File exists: 'behaviour_charts'


In [3]:
def load_triples_data() -> Tuple[dict, dict]:
    samples = {m: {ds: {} for ds in ds_list} for m in model_list}
    summaries = {m: {ds: {} for ds in ds_list} for m in model_list}
    for model, ds in zip(all_models, all_ds):
        base_fp = Path("0_stats", "similarity_triples")
        samples[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "samples.json")
        )
        summaries[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "summary.json")
        )

    return samples, summaries

Loading eval responses for correlation between triple similarity and performance

In [4]:
def load_eval_responses():
    data = {m: {ds: {}} for m in model_list for ds in ds_list}
    for model, ds in zip(all_models, all_ds):
        eval_hdl.set_curr_model(model)
        eval_hdl.set_curr_ds(ds)
        data[model][ds] = data_io.load_json(Path("..", eval_hdl.get_eval_path()))
    return data


eval_data = load_eval_responses()

In [5]:
samples, summaries = load_triples_data()
for m, ds in zip(all_models, all_ds):
    print(f">> Results for {m.upper()} on {ds.upper()}")
    print(f"- Num. entries: {len(samples[m][ds])}")
    print(f"- Avg. SIM: {summaries[m][ds]['cosine_avg']:.3f}")
    print(f"- Std. SIM: {summaries[m][ds]['cosine_std']:.3f}")
    print("-" * 30)

>> Results for INTERNVL2 on LLAVA-BENCH
- Num. entries: 58
- Avg. SIM: 0.455
- Std. SIM: 0.126
------------------------------
>> Results for INTERNVL2 on MMBENCH
- Num. entries: 139
- Avg. SIM: 0.495
- Std. SIM: 0.124
------------------------------
>> Results for INTERNVL2 on SEED
- Num. entries: 150
- Avg. SIM: 0.478
- Std. SIM: 0.140
------------------------------
>> Results for INTERNVL2 on VQAV2
- Num. entries: 150
- Avg. SIM: 0.552
- Std. SIM: 0.146
------------------------------
>> Results for LLAVA-1.6 on LLAVA-BENCH
- Num. entries: 44
- Avg. SIM: 0.461
- Std. SIM: 0.138
------------------------------
>> Results for LLAVA-1.6 on MMBENCH
- Num. entries: 133
- Avg. SIM: 0.534
- Std. SIM: 0.167
------------------------------
>> Results for LLAVA-1.6 on SEED
- Num. entries: 129
- Avg. SIM: 0.485
- Std. SIM: 0.148
------------------------------
>> Results for LLAVA-1.6 on VQAV2
- Num. entries: 142
- Avg. SIM: 0.513
- Std. SIM: 0.138
------------------------------
>> Results for MINIG

In [6]:
# Check the range of cosine similarity values obtained
for m, ds in zip(all_models, all_ds):
    print("=" * 70)
    print(f"Ranges for {m.upper()} on {ds.upper()}")
    print("\t\t == Min. == \t\t == Max. == \t Support")
    # First, get sample-level averages
    mes = [v["avg"] for v in samples[m][ds].values()]
    min_mes = np.min(mes).item()
    max_mes = np.max(mes).item()
    print(f"Sample-level \t {min_mes:.4f} \t\t {max_mes:.4f} \t {len(mes)}")

    # Then, consider the individual triples
    mes = [t["cosine"] for v in samples[m][ds].values() for t in v["measures"]]
    min_mes = np.min(mes).item()
    max_mes = np.max(mes).item()
    print(f"Triple-level \t {min_mes:.4f} \t\t {max_mes:.4f} \t {len(mes)}")

    print("-" * 70)
    negs = [m for m in mes if m <= 0]
    print(f"Num. negative triple similarities: {len(negs)}/{len(mes)}")
print("=" * 70)

Ranges for INTERNVL2 on LLAVA-BENCH
		 == Min. == 		 == Max. == 	 Support
Sample-level 	 0.2014 		 0.6784 	 58
Triple-level 	 0.0301 		 0.9140 	 565
----------------------------------------------------------------------
Num. negative triple similarities: 0/565
Ranges for INTERNVL2 on MMBENCH
		 == Min. == 		 == Max. == 	 Support
Sample-level 	 0.1587 		 0.8050 	 139
Triple-level 	 -0.0282 		 1.0000 	 1287
----------------------------------------------------------------------
Num. negative triple similarities: 1/1287
Ranges for INTERNVL2 on SEED
		 == Min. == 		 == Max. == 	 Support
Sample-level 	 0.1496 		 0.9780 	 150
Triple-level 	 -0.0557 		 1.0000 	 1254
----------------------------------------------------------------------
Num. negative triple similarities: 2/1254
Ranges for INTERNVL2 on VQAV2
		 == Min. == 		 == Max. == 	 Support
Sample-level 	 0.2376 		 0.9209 	 150
Triple-level 	 0.0343 		 1.0000 	 1014
----------------------------------------------------------------------
Num.

We have very few triple similarities that are slightly negative, let's clip those to `0.0`.

Mapping triples to behaviour types

In [7]:
df = pd.DataFrame()
for m, ds in zip(all_models, all_ds):
    for id_q in samples[m][ds]:
        # We have very few cosine similarities that are sligthly negative.
        # Clip those to 0.0. They are basically irrelevant.
        cosine_vals = [
            m["cosine"] if m["cosine"] >= 0.0 else 0.0
            for m in samples[m][ds][id_q]["measures"]
        ]
        row = {
            "question_id": int(id_q),
            "model": m,
            "dataset": ds,
            "similarity": [cosine_vals],
        }
        temp_df = pd.DataFrame.from_dict(row)
        df = pd.concat([df, temp_df], ignore_index=True)

In [8]:
for _, row in df.iterrows():
    if np.any(np.array(row["similarity"]) < 0.0):
        print(row)

In [9]:
df.head()

,question_id,model,dataset,similarity
0,0,internvl2,llava-bench,"[0.4406552016735077, 0.5663555860519409, 0.597..."
1,1,internvl2,llava-bench,"[0.5914754867553711, 0.7528765201568604, 0.653..."
2,2,internvl2,llava-bench,"[0.5391669273376465, 0.5219969153404236, 0.502..."
3,3,internvl2,llava-bench,"[0.44364720582962036, 0.4439797103404999, 0.45..."
4,4,internvl2,llava-bench,"[0.3434138894081116, 0.30635470151901245, 0.38..."


Add model performance to dataframe

In [10]:
# Data preparation
eval_data_clean = {model: {ds: [] for ds in eval_data[model]} for model in eval_data}
for model, ds in zip(all_models, all_ds):
    if ds in OPEN_DATASETS:
        ranges = list(eval_data[model][ds]["ranges"].keys())
        for r in ranges:
            eval_data_clean[model][ds].extend(
                [
                    {
                        "question_id": int(d["question_id"]),
                        "bertscore_f1": d["measures"]["f1"],
                    }
                    for d in eval_data[model][ds]["ranges"][r]
                ]
            )
        # Parse remaning, invalid responses
        eval_data_clean[model][ds].extend(
            [
                {"question_id": int(d["question_id"]), "bertscore_f1": 0.0}
                for d in eval_data[model][ds]["invalid_resp"]
            ]
        )
    elif ds in MC_DATASETS:
        # Parse correct_resp and wrong_resp
        eval_data_clean[model][ds].extend(
            [
                {"question_id": int(d["question_id"]), "correct": 1}
                for d in eval_data[model][ds]["correct_resp"]
            ]
        )
        eval_data_clean[model][ds].extend(
            [
                {"question_id": int(d["question_id"]), "correct": 0}
                for d in eval_data[model][ds]["wrong_resp"]
            ]
        )
        # Parse remaning, invalid responses
        eval_data_clean[model][ds].extend(
            [
                {"question_id": int(d["question_id"]), "correct": 0}
                for d in eval_data[model][ds]["invalid_resp"]
            ]
        )
    else:
        print("Error")

In [11]:
# Quick count to check that everything is there
for model, ds in zip(all_models, all_ds):
    print(f"{model.upper()} on {ds.upper()}: {len(eval_data_clean[model][ds])}")

INTERNVL2 on LLAVA-BENCH: 60
INTERNVL2 on MMBENCH: 139
INTERNVL2 on SEED: 150
INTERNVL2 on VQAV2: 150
LLAVA-1.6 on LLAVA-BENCH: 60
LLAVA-1.6 on MMBENCH: 139
LLAVA-1.6 on SEED: 150
LLAVA-1.6 on VQAV2: 150
MINIGPT4 on LLAVA-BENCH: 60
MINIGPT4 on MMBENCH: 139
MINIGPT4 on SEED: 150
MINIGPT4 on VQAV2: 150
SHAREGPT4V on LLAVA-BENCH: 60
SHAREGPT4V on MMBENCH: 139
SHAREGPT4V on SEED: 150
SHAREGPT4V on VQAV2: 150


In [12]:
df[df["dataset"] == "seed"].head()

,question_id,model,dataset,similarity
197,3151,internvl2,seed,"[0.5638384819030762, 0.544086754322052, 0.5812..."
198,1656,internvl2,seed,"[0.623802900314331, 0.6332287192344666, 0.7028..."
199,154,internvl2,seed,"[0.608259916305542, 0.651092529296875, 0.60042..."
200,3057,internvl2,seed,"[0.5762706995010376, 0.7283714413642883, 0.485..."
201,2173,internvl2,seed,"[0.574478805065155, 0.5702104568481445, 0.5464..."


In [13]:
def append_outcome(row, eval_data_clean):
    data = next(
        (
            e
            for e in eval_data_clean[row["model"]][row["dataset"]]
            if e["question_id"] == row["question_id"]
        ),
        None,
    )
    if data:
        if row["dataset"] in MC_DATASETS:
            return data["correct"]
        elif row["dataset"] in OPEN_DATASETS:
            return data["bertscore_f1"]
        else:
            return None
    else:
        return None


def get_outcome(row, eval_data_clean):
    data = next(
        (
            e
            for e in eval_data_clean[row["model"]][row["dataset"]]
            if e["question_id"] == row["question_id"]
        ),
        None,
    )
    return data


df["oe_outcome"] = df.apply(
    lambda x: get_outcome(x, eval_data_clean)["bertscore_f1"]
    if x["dataset"] in OPEN_DATASETS
    else None,
    axis=1,
)
df["mc_outcome"] = df.apply(
    lambda x: get_outcome(x, eval_data_clean)["correct"]
    if x["dataset"] in MC_DATASETS
    else None,
    axis=1,
)
df.head()

,question_id,model,dataset,similarity,oe_outcome,mc_outcome
0,0,internvl2,llava-bench,"[0.4406552016735077, 0.5663555860519409, 0.597...",0.792572,NaN
1,1,internvl2,llava-bench,"[0.5914754867553711, 0.7528765201568604, 0.653...",0.623514,NaN
2,2,internvl2,llava-bench,"[0.5391669273376465, 0.5219969153404236, 0.502...",0.623918,NaN
3,3,internvl2,llava-bench,"[0.44364720582962036, 0.4439797103404999, 0.45...",0.609265,NaN
4,4,internvl2,llava-bench,"[0.3434138894081116, 0.30635470151901245, 0.38...",0.590468,NaN


In [14]:
df[df["dataset"] == "seed"].head()

,question_id,model,dataset,similarity,oe_outcome,mc_outcome
197,3151,internvl2,seed,"[0.5638384819030762, 0.544086754322052, 0.5812...",NaN,1.0
198,1656,internvl2,seed,"[0.623802900314331, 0.6332287192344666, 0.7028...",NaN,1.0
199,154,internvl2,seed,"[0.608259916305542, 0.651092529296875, 0.60042...",NaN,1.0
200,3057,internvl2,seed,"[0.5762706995010376, 0.7283714413642883, 0.485...",NaN,1.0
201,2173,internvl2,seed,"[0.574478805065155, 0.5702104568481445, 0.5464...",NaN,1.0


In [15]:
for model, ds in zip(all_models, all_ds):
    subdf = df[(df["model"] == model) & (df["dataset"] == ds)]
    measures = []
    # measures.extend(row["similarity"] for _, row in subdf.iterrows())
    for _, row in subdf.iterrows():
        measures.extend(row["similarity"])
    print(f"{model.upper()} on {ds.upper()}: {len(measures)}")

INTERNVL2 on LLAVA-BENCH: 565
INTERNVL2 on MMBENCH: 1287
INTERNVL2 on SEED: 1254
INTERNVL2 on VQAV2: 1014
LLAVA-1.6 on LLAVA-BENCH: 279
LLAVA-1.6 on MMBENCH: 823
LLAVA-1.6 on SEED: 853
LLAVA-1.6 on VQAV2: 939
MINIGPT4 on LLAVA-BENCH: 525
MINIGPT4 on MMBENCH: 1032
MINIGPT4 on SEED: 1378
MINIGPT4 on VQAV2: 1194
SHAREGPT4V on LLAVA-BENCH: 279
SHAREGPT4V on MMBENCH: 645
SHAREGPT4V on SEED: 241
SHAREGPT4V on VQAV2: 46


Current dataframe
- `question_id`
- `model`
- `dataset`
- `similarity`: list of similarity vals for each RK for that sample against all of SKs
- `oe_outcome`: BERTScore F1
- `mc_outcome`: correct (1.0) or incorrect (0.0)

In [16]:
prec = 6
step = round(1 / 3, prec)
cosine_ranges = {
    idx: {"start": float(round(val, prec)), "end": float(round(val + step, prec))}
    for idx, val in enumerate(np.linspace(0, 1, 4))
}

# remove last range, meaningless here
cosine_ranges.pop(3)
cosine_ranges[2]["end"] = 1.0
cosine_ranges

{0: {'start': 0.0, 'end': 0.333333},
 1: {'start': 0.333333, 'end': 0.666666},
 2: {'start': 0.666667, 'end': 1.0}}

In [17]:
counts_type = {idx: {} for idx in cosine_ranges}

for idx, r in cosine_ranges.items():
    for m, ds in zip(all_models, all_ds):
        if m not in counts_type[idx]:
            counts_type[idx][m] = {}
        if ds not in counts_type[idx][m]:
            counts_type[idx][m][ds] = 0

        subdf = df[(df["dataset"] == ds) & (df["model"] == m)]
        for _, row in subdf.iterrows():
            for sim in row["similarity"]:
                temp = float(round(sim, prec))
                if temp >= r["start"] and temp <= r["end"]:
                    counts_type[idx][m][ds] += 1

            # counts_type[idx][m][ds] += np.count_nonzero(
            #     (np.array(row["similarity"]) >= r["start"])
            #     & (np.array(row["similarity"]) <= r["end"])
            # )

In [18]:
for model, ds in zip(all_models, all_ds):
    curr = 0
    for t in counts_type:
        curr += counts_type[t][model][ds]
    print(f"{model.upper()} on {ds.upper()}: {curr}")

INTERNVL2 on LLAVA-BENCH: 565
INTERNVL2 on MMBENCH: 1287
INTERNVL2 on SEED: 1254
INTERNVL2 on VQAV2: 1014
LLAVA-1.6 on LLAVA-BENCH: 279
LLAVA-1.6 on MMBENCH: 823
LLAVA-1.6 on SEED: 853
LLAVA-1.6 on VQAV2: 939
MINIGPT4 on LLAVA-BENCH: 525
MINIGPT4 on MMBENCH: 1032
MINIGPT4 on SEED: 1378
MINIGPT4 on VQAV2: 1194
SHAREGPT4V on LLAVA-BENCH: 279
SHAREGPT4V on MMBENCH: 645
SHAREGPT4V on SEED: 241
SHAREGPT4V on VQAV2: 46


In [19]:
counts_type

{0: {'internvl2': {'llava-bench': 158,
   'mmbench': 296,
   'seed': 310,
   'vqav2': 230},
  'llava-1.6': {'llava-bench': 77, 'mmbench': 220, 'seed': 241, 'vqav2': 251},
  'minigpt4': {'llava-bench': 221, 'mmbench': 419, 'seed': 499, 'vqav2': 335},
  'sharegpt4v': {'llava-bench': 87, 'mmbench': 104, 'seed': 46, 'vqav2': 6}},
 1: {'internvl2': {'llava-bench': 341,
   'mmbench': 740,
   'seed': 749,
   'vqav2': 530},
  'llava-1.6': {'llava-bench': 163, 'mmbench': 454, 'seed': 482, 'vqav2': 480},
  'minigpt4': {'llava-bench': 258, 'mmbench': 466, 'seed': 750, 'vqav2': 639},
  'sharegpt4v': {'llava-bench': 159,
   'mmbench': 403,
   'seed': 132,
   'vqav2': 25}},
 2: {'internvl2': {'llava-bench': 66,
   'mmbench': 251,
   'seed': 195,
   'vqav2': 254},
  'llava-1.6': {'llava-bench': 39, 'mmbench': 149, 'seed': 130, 'vqav2': 208},
  'minigpt4': {'llava-bench': 46, 'mmbench': 147, 'seed': 129, 'vqav2': 220},
  'sharegpt4v': {'llava-bench': 33, 'mmbench': 138, 'seed': 63, 'vqav2': 15}}}

In [20]:
counts_plot = {idx: {} for idx in cosine_ranges}

for idx, r in cosine_ranges.items():
    for m, ds in zip(all_models, all_ds):
        if ds not in counts_plot[idx]:
            counts_plot[idx][ds] = []

        subdf = df[(df["dataset"] == ds) & (df["model"] == m)]
        curr_count = 0
        for _, row in subdf.iterrows():
            for sim in row["similarity"]:
                temp = float(round(sim, prec))
                if temp >= r["start"] and temp <= r["end"]:
                    curr_count += 1
        counts_plot[idx][ds].append(curr_count)

# Aligning the data for plotting
# Swap: 0 (Type 3 failures) <-> 2 (Type 1 failures)
# Good: 1 (Type 2 failures)

temp = counts_plot[0]
counts_plot[0] = counts_plot[2]
counts_plot[2] = temp

In [21]:
counts_plot

{0: {'llava-bench': [66, 39, 46, 33],
  'mmbench': [251, 149, 147, 138],
  'seed': [195, 130, 129, 63],
  'vqav2': [254, 208, 220, 15]},
 1: {'llava-bench': [341, 163, 258, 159],
  'mmbench': [740, 454, 466, 403],
  'seed': [749, 482, 750, 132],
  'vqav2': [530, 480, 639, 25]},
 2: {'llava-bench': [158, 77, 221, 87],
  'mmbench': [296, 220, 419, 104],
  'seed': [310, 241, 499, 46],
  'vqav2': [230, 251, 335, 6]}}

In [22]:
out_path = Path(CHARTS_DIR, "triple_type_counts.pdf")
vis.plot_behaviour_counts(counts_plot, out_path)

/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:447: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


TODO: analysis number of behaviours of a certain type vs. model performance